In [ ]:
pip install pandas

In [ ]:
from numpy import shape
import requests
import pandas as pd
from datetime import datetime

url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

all_records = []
start_year = datetime.now().year - 5   # Last five years
end_year = datetime.now().year

for year in range(start_year, end_year + 1):
    for month in range(1, 13):
        start_date = f"{year}-{month:02d}-01"
        if month == 12:
            end_date = f"{year+1}-01-01"
        else:
            end_date = f"{year}-{month+1:02d}-01"

        params = {
            "format": "geojson",
            "starttime": start_date,
            "endtime": end_date,
            "minmagnitude": 3
        }
        response = requests.get(url, params=params)
        if response.status_code != 200:
            print(f"⚠️ Failed for {start_date}: {response.text[:200]}")
            continue

        try:
            data = response.json()
        except Exception as e:
            print(f"⚠️ JSON error for {start_date}: {e}")
            continue

        features = data.get("features", [])
        for f in features:
            p = f.get("properties", {})
            g = f.get("geometry", {}).get("coordinates", [])
            all_records.append({
                "id": f.get("id"),
                "time": pd.to_datetime(p.get("time"), unit="ms") if p.get("time") else None,
                "updated": pd.to_datetime(p.get("updated"), unit="ms") if p.get("updated") else None,
                "latitude": g[1] if len(g) > 1 else None,
                "longitude": g[0] if len(g) > 0 else None,
                "depth_km": g[2] if len(g) > 2 else None,
                "mag": p.get("mag"),
                "magType": p.get("magType"),
                "place": p.get("place"),
                "status": p.get("status"),
                "tsunami": p.get("tsunami"),
                "alert": p.get("alert"),
                "felt": p.get("felt"),
                "cdi": p.get("cdi"),
                "mmi": p.get("mmi"),
                "sig": p.get("sig"),
                "net": p.get("net"),
                "code": p.get("code"),
                "ids": p.get("ids"),
                "sources": p.get("sources"),
                "types": p.get("types"),
                "nst": p.get("nst"),
                "dmin": p.get("dmin"),
                "rms": p.get("rms"),
                "gap": p.get("gap"),
                "type": p.get("type")
            })

df = pd.DataFrame(all_records)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
if not df.empty:
    print(df.head())

    df.to_csv ("earthquakes_raw.csv", index=False)

    print(df.shape)



Rows: 122183
Columns: 26
           id                    time                 updated  latitude  \
0  us6000ddi8 2021-01-31 23:20:49.923 2021-04-16 19:02:44.040  -31.7493   
1  us6000dev6 2021-01-31 23:08:17.161 2021-04-16 19:03:47.040  -15.4902   
2  us6000dev5 2021-01-31 22:54:19.760 2021-04-16 19:03:47.040   19.7529   
3  us6000ddhs 2021-01-31 22:06:00.832 2021-04-16 19:02:43.040   28.1524   
4  us6000dev4 2021-01-31 21:51:14.016 2021-04-16 19:03:46.040   71.3212   

   longitude  depth_km  mag magType  \
0   -68.9337     17.27  4.7     mwr   
1  -177.2052    426.71  4.1      mb   
2   121.3159     46.73  4.7      mb   
3    57.2570     10.00  4.9      mb   
4    -3.7578     10.00  4.0      mb   

                                               place    status  ...  net  \
0        29 km SW of Villa Basilio Nievas, Argentina  reviewed  ...   us   
1                                        Fiji region  reviewed  ...   us   
2                    103 km SW of Basco, Philippines  reviewe

In [ ]:
 df.isnull().sum()

,0
id,0
time,0
updated,0
latitude,0
longitude,0
depth_km,0
mag,0
magType,0
place,0
status,0


In [ ]:
df['rms'].mean()

np.float64(0.6613179222504681)

In [ ]:
df[df['rms'].isnull()]

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,...,net,code,ids,sources,types,nst,dmin,rms,gap,type
1313,iscgem619646626,2021-01-09 03:38:16.140,2025-12-22 18:48:10.225,-45.9700,95.8170,12.00,5.0500,mw,southeast Indian Ridge,reviewed,...,iscgem,619646626,",iscgem619646626,",",iscgem,",",origin,",NaN,NaN,NaN,NaN,earthquake
1764,iscgem619932863,2021-02-28 06:11:33.050,2025-12-22 18:53:26.421,-15.4140,-176.2150,26.00,5.1400,mw,"235 km S of Mata-Utu, Wallis and Futuna",reviewed,...,iscgem,619932863,",iscgem619932863,",",iscgem,",",origin,",NaN,NaN,NaN,NaN,earthquake
1769,iscgem619932856,2021-02-28 03:31:23.460,2025-12-22 18:00:20.298,-15.6850,-176.0140,26.00,5.1600,mw,"239 km W of Hihifo, Tonga",reviewed,...,iscgem,619932856,",iscgem619932856,",",iscgem,",",origin,",NaN,NaN,NaN,NaN,earthquake
2356,iscgem619900745,2021-02-17 04:57:34.290,2025-12-22 18:51:55.980,1.1480,-84.9510,38.00,5.0600,mw,off the coast of Ecuador,reviewed,...,iscgem,619900745,",iscgem619900745,",",iscgem,",",origin,",NaN,NaN,NaN,NaN,earthquake
4506,iscgem620210242,2021-03-14 14:09:13.950,2025-12-22 18:55:19.233,-59.3030,-31.0410,10.00,5.1800,mw,Scotia Sea,reviewed,...,iscgem,620210242,",iscgem620210242,",",iscgem,",",origin,",NaN,NaN,NaN,NaN,earthquake
5134,iscgem620177767,2021-03-07 03:28:18.110,2025-12-22 18:00:29.265,-28.8440,-176.8060,36.00,5.1000,mw,Kermadec Islands region,reviewed,...,iscgem,620177767,",iscgem620177767,",",iscgem,",",origin,",NaN,NaN,NaN,NaN,earthquake
5225,iscgem619921848,2021-03-06 17:32:59.220,2025-12-22 18:52:42.736,-28.4610,-176.3120,19.10,5.5000,mw,Kermadec Islands region,reviewed,...,iscgem,619921848,",usauto7000dgk3,iscgem619921848,",",usauto,iscgem,",",internal-moment-tensor,origin,",NaN,NaN,NaN,NaN,earthquake
5477,iscgem620360384,2021-03-05 12:28:52.720,2025-12-22 18:58:56.666,-28.7790,-176.6900,20.30,5.1700,mw,Kermadec Islands region,reviewed,...,iscgem,620360384,",iscgem620360384,",",iscgem,",",origin,",NaN,NaN,NaN,NaN,earthquake
7856,iscgem620441036,2021-05-25 23:20:50.420,2025-12-22 19:01:04.481,-41.3650,-89.1120,10.00,5.2400,mw,southeast of Easter Island,reviewed,...,iscgem,620441036,",iscgem620441036,",",iscgem,",",origin,",NaN,NaN,NaN,NaN,earthquake
8193,iscgem620505071,2021-05-21 06:01:37.380,2025-12-22 19:01:51.484,-35.7680,-103.7930,10.00,5.0700,mw,southeast of Easter Island,reviewed,...,iscgem,620505071,",iscgem620505071,",",iscgem,",",origin,",NaN,NaN,NaN,NaN,earthquake


In [ ]:
df.rms

,rms
0,0.82
1,0.29
2,0.69
3,0.61
4,0.50
...,...
121961,0.75
121962,0.93
121963,1.32
121964,0.80


In [ ]:
df.alert

,alert
0,None
1,None
2,None
3,None
4,None
...,...
121961,None
121962,None
121963,None
121964,None


In [ ]:
df.felt

,felt
0,12.0
1,NaN
2,NaN
3,NaN
4,NaN
...,...
121961,NaN
121962,3.0
121963,NaN
121964,NaN


In [ ]:
df.cdi

,cdi
0,3.7
1,NaN
2,NaN
3,NaN
4,NaN
...,...
121961,NaN
121962,2.7
121963,NaN
121964,NaN


In [ ]:
df.mmi

,mmi
0,NaN
1,NaN
2,NaN
3,NaN
4,NaN
...,...
121961,NaN
121962,NaN
121963,NaN
121964,NaN


In [ ]:
df.nst

,nst
0,NaN
1,NaN
2,NaN
3,NaN
4,NaN
...,...
121961,40.0
121962,201.0
121963,31.0
121964,49.0


In [ ]:
df.dmin

,dmin
0,0.2940
1,1.4710
2,3.0570
3,3.3300
4,6.0230
...,...
121961,5.4730
121962,1.6210
121963,14.9720
121964,16.9060


In [ ]:
df.rms

,rms
0,0.82
1,0.29
2,0.69
3,0.61
4,0.50
...,...
121961,0.75
121962,0.93
121963,1.32
121964,0.80


In [ ]:
df.gap

,gap
0,42.0
1,64.0
2,106.0
3,71.0
4,65.0
...,...
121961,58.0
121962,24.0
121963,152.0
121964,99.0


In [ ]:
df['gap'].mean()

np.float64(122.86458919761981)

In [ ]:
df.alert

,alert
0,None
1,None
2,None
3,None
4,None
...,...
122108,None
122109,None
122110,None
122111,None


In [ ]:
df.to_csv("earthquakes_raw.csv", index=False)

print(df.shape)

(122113, 26)


In [ ]:
import os

print("Current Directory:")
print(os.getcwd())

df.to_csv("earthquakes_raw.csv", index=False)


Current Directory:
/content


In [ ]:
print(df.info())
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122116 entries, 0 to 122115
Data columns (total 26 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   id         122116 non-null  object        
 1   time       122116 non-null  datetime64[ns]
 2   updated    122116 non-null  datetime64[ns]
 3   latitude   122116 non-null  float64       
 4   longitude  122116 non-null  float64       
 5   depth_km   122116 non-null  float64       
 6   mag        122116 non-null  float64       
 7   magType    122116 non-null  object        
 8   place      122116 non-null  object        
 9   status     122116 non-null  object        
 10  tsunami    122116 non-null  int64         
 11  alert      4986 non-null    object        
 12  felt       18512 non-null   float64       
 13  cdi        18512 non-null   float64       
 14  mmi        10844 non-null   float64       
 15  sig        122116 non-null  int64         
 16  net        122116 no